
# Predicting Publisher
**Author:** AARON 
**Date:** April 6, 2026 
**Objective:** Create a Randome Forest Model to predict Publisher for articles.




## Introduction
- Using the Random Forest Model Predict which Publisher wrote an article.

# Basic Model: Random Forest

This section builds the **Basic Model** for publisher prediction using only the selected sentiment score inputs.

## Included inputs

### Keyword score inputs
- `iran_score`
- `iranian_score`
- `israel_israeli_score`
- `hormuz_score`
- `supreme_leader_score`
- `tehran_score`
- `trump_score`
- `us_united_states_score`

### Text score inputs
- `title_score`
- `full_text_score`
- `start_text_score`
- `half_text_score`

## Excluded inputs
No other columns are used in this model.

## Model choice
A **Random Forest Classifier** is used as the baseline model. It can capture nonlinear relationships and interactions between the sentiment score inputs without requiring complex feature engineering.

In [1]:
# Basic Model using Random Forest
# Only the selected score columns are used as inputs

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv("../code/iran_war_complete_articles_sentiment.csv")

# -----------------------------
# Define target column
# -----------------------------
target_col = "publisher"

# -----------------------------
# Define Basic Model feature columns
# -----------------------------
basic_features = [
    "iran_score",
    "iranian_score",
    "israel_israeli_score",
    "hormuz_score",
    "supreme_leader_score",
    "tehran_score",
    "trump_score",
    "us_united_states_score",
    "title_score",
    "full_text_score",
    "start_text_score",
    "half_text_score",
]

# -----------------------------
# Keep only required columns
# -----------------------------
df_basic = df[basic_features + [target_col]].copy()

print("Basic model columns:")
print(df_basic.columns.tolist())
print("\nDataset shape:", df_basic.shape)

Basic model columns:
['iran_score', 'iranian_score', 'israel_israeli_score', 'hormuz_score', 'supreme_leader_score', 'tehran_score', 'trump_score', 'us_united_states_score', 'title_score', 'full_text_score', 'start_text_score', 'half_text_score', 'publisher']

Dataset shape: (582, 13)


## Data preparation

This model uses only the selected numeric score columns.

Steps:
1. Remove rows with missing target labels
2. Fill missing feature values with 0
3. Split the data into training and testing sets

In [2]:
# Drop rows with missing target
# df_basic = df_basic.dropna(subset=[target_col]).copy()

X = df_basic[basic_features]
y = df_basic[target_col]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nMissing values in features:")
print(X.isnull().sum())

Feature shape: (582, 12)
Target shape: (582,)

Missing values in features:
iran_score                0
iranian_score             0
israel_israeli_score      0
hormuz_score              0
supreme_leader_score      0
tehran_score              0
trump_score               0
us_united_states_score    0
title_score               0
full_text_score           0
start_text_score          0
half_text_score           0
dtype: int64


In [3]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 465
Testing rows: 117


## Train the Random Forest model

A Random Forest model is trained using only the selected score inputs.

In [4]:
basic_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        (
            "clf",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

basic_model.fit(X_train, y_train)

y_pred = basic_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)

print("Basic Model Accuracy:", round(acc, 4))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Basic Model Accuracy: 0.3761

Classification Report:

              precision    recall  f1-score   support

  Al Jazeera       0.47      0.54      0.50        41
         BBC       0.22      0.16      0.18        32
    Fox News       0.25      0.22      0.24        18
    NBC News       0.42      0.50      0.46        26

    accuracy                           0.38       117
   macro avg       0.34      0.35      0.34       117
weighted avg       0.36      0.38      0.36       117



## Confusion matrix

The confusion matrix shows how often each publisher is predicted correctly and where misclassification happens.

In [5]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(cm, index=basic_model.classes_, columns=basic_model.classes_)

cm_df

,Al Jazeera,BBC,Fox News,NBC News
Al Jazeera,22,8,6,5
BBC,18,5,3,6
Fox News,2,5,4,7
NBC News,5,5,3,13


## Feature importance

Random Forest provides feature importance values, which help show which of the selected sentiment score inputs contribute most to the predictions.

In [6]:
importances = basic_model.named_steps["clf"].feature_importances_

feature_importance_df = pd.DataFrame(
    {"feature": basic_features, "importance": importances}
).sort_values(by="importance", ascending=False)

feature_importance_df

,feature,importance
7,us_united_states_score,0.108340
2,israel_israeli_score,0.107353
9,full_text_score,0.098596
8,title_score,0.098175
11,half_text_score,0.092796
0,iran_score,0.089877
10,start_text_score,0.087340
1,iranian_score,0.086001
3,hormuz_score,0.076279
6,trump_score,0.074668
